In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

from torchvision import transforms
import numpy as np
import pandas as pd
from PIL import Image

from tqdm.auto import tqdm
import os
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

# Verify environment
print(f'Pytorch Version: {torch.__version__}')
print(f'Cuda is available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Pytorch Version: 2.4.1
Cuda is available: False
Device: cpu


/Users/luule/miniconda3/envs/wbc/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Directory paths
DIR_PHASE_2_TEST = 'MACENKO_NORMALIZE/test'
LABEL_PATH_PHASE_2_TEST = 'wbc-isbi/phase2_test.csv'
SAVE_DIR = './checkpoints'

# Class information
NUM_CLASSES = 13
CLASS_NAMES = ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Training class distribution (for post-processing)
TRAIN_COUNTS = {
    'SNE': 17354, 'LY': 10801, 'MO': 3661, 'BL': 2683,
    'EO': 1148, 'MY': 588, 'BA': 554, 'BNE': 522,
    'VLY': 488, 'MMY': 480, 'PMY': 152, 'PC': 90, 'PLY': 14
}

# Model configs
config1 = {
    'model': {
        'name': 'resnet50',
        'pretrained': True,
        'num_classes': NUM_CLASSES,
        'head': {'type': 'custom', 'hidden_dim': 512, 'dropout': 0.0},
        'normalize': {'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)},
    },
    'data': {'image_size': 368, 'batch_size': 256, 'num_workers': 0},
}

config2 = {
    'model': {
        'name': 'resnet152',
        'pretrained': True,
        'num_classes': NUM_CLASSES,
        'head': {'type': 'custom', 'hidden_dim': 512, 'dropout': 0.0},
        'normalize': {'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)},
    },
    'data': {'image_size': 368, 'batch_size': 256, 'num_workers': 0},
}

CONFIGS = [
    ('resnet50', config1, os.path.join(SAVE_DIR, 'final_model_resnet50.pt')),
    ('resnet152', config2, os.path.join(SAVE_DIR, 'final_model_resnet152.pt')),
]

ENSEMBLE_WEIGHTS = [1.0, 1.0]

print(f"Models to ensemble: {[c[0] for c in CONFIGS]}")

Models to ensemble: ['resnet50', 'resnet152']


In [17]:
def get_in_features(model):
    if hasattr(model, 'fc'):
        return model.fc.in_features
    if hasattr(model, 'head'):
        return model.head.in_features
    raise ValueError("Cannot infer in_features")


def create_head(in_features, config):
    head_type = config['model']['head']['type']
    num_classes = config['model']['num_classes']

    if head_type == 'default':
        return nn.Linear(in_features, num_classes)
    elif head_type == 'custom':
        hidden_dim = config['model']['head']['hidden_dim']
        dropout_rate = config['model']['head']['dropout']
        return nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, num_classes),
        )


def replace_head(model, new_head):
    if hasattr(model, 'fc'):
        model.fc = new_head
    elif hasattr(model, 'head'):
        model.head = new_head


def create_model(config):
    model_name = config['model']['name']

    if model_name == 'resnet50':
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.DEFAULT)
    elif model_name == 'resnet152':
        from torchvision.models import resnet152, ResNet152_Weights
        model = resnet152(weights=ResNet152_Weights.DEFAULT)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    in_features = get_in_features(model)
    model._in_features = in_features
    head = create_head(in_features, config)
    replace_head(model, head)

    return model


def load_checkpoint(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint['model']

    # Handle DataParallel mismatch
    ckpt_has_module = any(k.startswith('module.') for k in state_dict.keys())
    model_has_module = isinstance(model, nn.DataParallel)

    if ckpt_has_module and not model_has_module:
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    elif not ckpt_has_module and model_has_module:
        state_dict = {f'module.{k}': v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    print(f"✅ Loaded: {checkpoint_path}")
    
    return model

In [18]:
ensemble_models = []
ensemble_model_names = []

for model_name, config, checkpoint_path in CONFIGS:
    print(f"\nLoading {model_name}...")
    
    model = create_model(config)
    model = model.to(device)
    model = load_checkpoint(model, checkpoint_path, device)
    model.eval()
    
    ensemble_models.append(model)
    ensemble_model_names.append(model_name)

print(f"\n✅ {len(ensemble_models)} models loaded!")


Loading resnet50...
✅ Loaded: ./checkpoints/final_model_resnet50.pt

Loading resnet152...
✅ Loaded: ./checkpoints/final_model_resnet152.pt

✅ 2 models loaded!


In [19]:
def get_tta_transforms(img_size, mean, std):
    normalize = transforms.Normalize(mean, std)
    resize = transforms.Resize((img_size, img_size))

    return [
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.Lambda(lambda x: x.rotate(90)), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.Lambda(lambda x: x.rotate(180)), transforms.ToTensor(), normalize]),
        transforms.Compose([resize, transforms.CenterCrop(224), transforms.Lambda(lambda x: x.rotate(270)), transforms.ToTensor(), normalize]),
    ]


class TTATestDataset(Dataset):
    def __init__(self, df, img_dir, tta_transforms):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tta_transforms = tta_transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['ID'])
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        images = torch.stack([t(image) for t in self.tta_transforms])
        return images, img_id


def run_tta_inference_single(model, dataloader, device, model_name=""):
    model.eval()
    all_probs, all_ids = [], []

    with torch.no_grad():
        for images_batch, ids in tqdm(dataloader, desc=f"TTA {model_name}", leave=False):
            B, N, C, H, W = images_batch.shape
            images_flat = images_batch.view(-1, C, H, W).to(device)

            with autocast():
                outputs = model(images_flat)
                probs = F.softmax(outputs, dim=1)

            probs = probs.view(B, N, -1).mean(dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_ids.extend(ids)

    return np.array(all_probs), all_ids


def run_ensemble_tta_inference(models, model_names, dataloader, device, weights=None):
    num_models = len(models)
    
    if weights is None:
        weights = [1.0 / num_models] * num_models
    else:
        total_weight = sum(weights)
        weights = [w / total_weight for w in weights]
    
    print(f"\n🔀 Ensemble with {num_models} models, weights: {weights}")
    
    all_model_probs = []
    all_ids = None
    
    for i, (model, name) in enumerate(zip(models, model_names)):
        print(f"\n[{i+1}/{num_models}] {name}")
        probs, ids = run_tta_inference_single(model, dataloader, device, name)
        all_model_probs.append(probs)
        if all_ids is None:
            all_ids = ids
    
    all_model_probs = np.array(all_model_probs)
    weights_arr = np.array(weights).reshape(-1, 1, 1)
    averaged_probs = (all_model_probs * weights_arr).sum(axis=0)
    
    return averaged_probs, all_ids

In [20]:
print("\n" + "="*55)
print("🔮 ENSEMBLE TTA INFERENCE")
print("="*55)

# Config
IMAGE_SIZE = config1['data']['image_size']
NORMALIZE_MEAN = config1['model']['normalize']['mean']
NORMALIZE_STD = config1['model']['normalize']['std']
BATCH_SIZE = config1['data']['batch_size']
NUM_WORKERS = config1['data']['num_workers']

# Load test data
df_test = pd.read_csv(LABEL_PATH_PHASE_2_TEST)[['ID']]
print(f"Test samples: {len(df_test)}")

# Setup TTA
tta_transforms = get_tta_transforms(IMAGE_SIZE, NORMALIZE_MEAN, NORMALIZE_STD)
print(f"TTA augmentations: {len(tta_transforms)}")

# Dataset & Loader
test_dataset = TTATestDataset(df_test, DIR_PHASE_2_TEST, tta_transforms)
test_loader = DataLoader(
    test_dataset,
    batch_size=max(1, BATCH_SIZE // len(tta_transforms)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Run ensemble inference
all_probs, all_ids = run_ensemble_tta_inference(
    models=ensemble_models,
    model_names=ensemble_model_names,
    dataloader=test_loader,
    device=device,
    weights=ENSEMBLE_WEIGHTS
)

print(f"\n✅ Ensemble inference done: {len(all_probs)} samples")


🔮 ENSEMBLE TTA INFERENCE
Test samples: 16477
TTA augmentations: 8

🔀 Ensemble with 2 models, weights: [0.5, 0.5]

[1/2] resnet50


KeyboardInterrupt: 

In [ ]:
def smart_rare_class_adjustment(probs, class_names, train_counts, class_to_idx, 
                                 min_ratio=0.3, confidence_threshold=0.005):
    preds = probs.argmax(axis=1).copy()
    num_samples = len(preds)
    total_train = sum(train_counts.values())
    
    print("\n🎯 SMART RARE CLASS ADJUSTMENT")
    
    adjustments = []
    for cls_name in class_names:
        cls_idx = class_to_idx[cls_name]
        train_ratio = train_counts.get(cls_name, 0) / total_train
        expected_count = int(num_samples * train_ratio)
        min_expected = max(1, int(expected_count * min_ratio))
        current_count = (preds == cls_idx).sum()
        
        if current_count < min_expected:
            adjustments.append({
                'class': cls_name, 'idx': cls_idx,
                'current': current_count, 'min_expected': min_expected,
                'need': min_expected - current_count,
            })
    
    adjustments.sort(key=lambda x: x['need'], reverse=True)
    
    for adj in adjustments:
        cls_name, cls_idx = adj['class'], adj['idx']
        need_more = adj['need']
        
        cls_probs = probs[:, cls_idx].copy()
        cls_probs[preds == cls_idx] = -1
        
        candidates = [(i, cls_probs[i]) for i in range(num_samples) if cls_probs[i] > confidence_threshold]
        candidates.sort(key=lambda x: x[1], reverse=True)
        
        for idx, _ in candidates[:need_more]:
            preds[idx] = cls_idx
        
        print(f"   {cls_name}: {adj['current']} → {(preds == cls_idx).sum()}")
    
    return preds


def print_distribution(preds, class_names, class_to_idx, title="Distribution"):
    """Print prediction distribution for all classes."""
    print(f"\n{'='*50}")
    print(f"📊 {title}")
    print(f"{'='*50}")
    print(f"{'Class':<8} {'Count':>8} {'Percent':>10}")
    print(f"{'-'*30}")
    
    total = len(preds)
    for cls_name in class_names:
        cls_idx = class_to_idx[cls_name]
        count = (preds == cls_idx).sum()
        pct = count / total * 100
        print(f"{cls_name:<8} {count:>8} {pct:>9.2f}%")
    
    print(f"{'-'*30}")
    print(f"{'Total':<8} {total:>8}")
    print(f"{'='*50}")


# Print distribution BEFORE adjustment
baseline_preds = all_probs.argmax(axis=1)
print_distribution(baseline_preds, CLASS_NAMES, CLASS_TO_IDX, "BEFORE POST-PROCESSING")

# Apply post-processing
final_preds = smart_rare_class_adjustment(
    probs=all_probs,
    class_names=CLASS_NAMES,
    train_counts=TRAIN_COUNTS,
    class_to_idx=CLASS_TO_IDX,
    min_ratio=0.2,
    confidence_threshold=0.001,
)

# Print distribution AFTER adjustment
print_distribution(final_preds, CLASS_NAMES, CLASS_TO_IDX, "AFTER POST-PROCESSING")

# Save submission
pred_labels = [IDX_TO_CLASS[p] for p in final_preds]
submission = pd.DataFrame({'ID': all_ids, 'labels': pred_labels})
submission.to_csv('submission.csv', index=False)

print(f"\n✅ Saved: submission.csv")
print(f"🎉 INFERENCE COMPLETE!")